# Model 2: ESPN Media Scrapping
For this part of the model, we try to provide a sentiment score of the potential NBA all-star players. We use nltk to provide the sentiment score given a article title.

###**Overview / Procedure:**
1. Import all players for each season (19-20, 20-21 21-22, 22-23, 23-24)
2. Scrape 10 ESPN articles for each player per season using Google Custom API Searh and save in csv
3. Conduct Sentiment analysis for each player based on the 10 articles
4. Calculate a final Sentiment Analysis Score and save in csv
- S (Sentiment component) = the avg sentiment score of the 10 articles
- P (Popularity component) = the total search result returned by each query (player name)
- ALPHA = 0.91 Calculated through grid search
- final_sentiment_score = ALPHA * popularity_component + (1 - ALPHA) * sentiment_component

Training (Season: 19-20, 20-21, 21-22, 22-23) / Testing (Season: 23-24)

## Imports and API_KEY:

In [ ]:
import os
import requests
import pandas as pd
import re
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import numpy as np
from google.colab import files
import time

API_KEY_Isaac_2 = os.environ["GOOGLE_API_KEY"]  # set GOOGLE_API_KEY before running

# define CX to limit the search to ESPN articles only
CX = "00957679c400e4056"

## Read CSV Files and Set Players List
Reads the file import of players name for seasons 21-22, 22-23, 23-24 and define the list of players name that will be used for sentiment analsysis.

In [ ]:
# training 4 seasons (20, 21, 22, 23)
all_players_19_20 = pd.read_csv("2019-2020.csv")
all_players_20_21 = pd.read_csv("2020-2021.csv")
all_players_21_22 = pd.read_csv("2021-2022.csv")
all_players_22_23 = pd.read_csv("2022-2023.csv")
# test 1 season (24)
all_players_23_24 = pd.read_csv("2023-2024.csv")
# extra
all_players_25_26 = pd.read_csv("2025-2026.csv")


list_player_19_20 = all_players_19_20["Player"].unique().tolist()
list_player_20_21 = all_players_20_21["Player"].unique().tolist()
list_player_21_22 = all_players_21_22["Player"].unique().tolist()
list_player_22_23 = all_players_22_23["Player"].unique().tolist()
list_player_23_24 = all_players_23_24["Player"].unique().tolist()
list_player_25_26 = all_players_25_26["Player"].unique().tolist()

print(len(list_player_19_20))
print(len(list_player_20_21))
print(len(list_player_21_22))
print(len(list_player_22_23))
print(len(list_player_23_24))
print(len(list_player_25_26))

# check output
# print(all_players_21_22.head())
# print(list_player_23_24)


530
541
606
540
573
491


Set All Star Players

In [ ]:
all_stars_23_24 = ['Giannis Antetokounmpo', 'Bam Adebayo', 'Tyrese Haliburton', 'Damian Lillard', 'Jayson Tatum', 'Paolo Banchero',
                'Jaylen Brown', 'Jalen Brunson', 'Tyrese Maxey', 'Donovan Mitchell', 'Julius Randle', 'Joel Embiid',
                'LeBron James', 'Luka Dončić', 'Kevin Durant', 'Shai Gilgeous-Alexander', 'Nikola Jokić', 'Devin Booker', 'Stephen Curry', 'Anthony Davis',
                'Anthony Edwards', 'Paul George', 'Kawhi Leonard', 'Karl-Anthony Towns']


all_stars_22_23 = ['Giannis Antetokounmpo', 'Lauri Markkanen', 'Donovan Mitchell', 'Ja Morant', 'Jayson Tatum', 'Bam Adebayo',
                'Shai Gilgeous-Alexander', 'DeMar DeRozan', 'Jrue Holiday', 'Damian Lillard', 'Domantas Sabonis', 'Kevin Durant',
                'LeBron James', 'Luka Dončić', 'Joel Embiid', 'Kyrie Irving', 'Nikola Jokić', 'Jaylen Brown', 'Stephen Curry', 'Paul George',
                'Tyrese Haliburton', 'Jaren Jackson Jr.', 'Julius Randle', 'Zion Williamson']


all_stars_21_22 = ['Kevin Durant', 'Joel Embiid', 'Ja Morant', 'Jayson Tatum', 'Andrew Wiggins', 'Trae Young',
                'Devin Booker', 'Rudy Gobert', 'Zach LaVine', 'Khris Middleton', 'Karl-Anthony Towns', 'Draymond Green',
                'LeBron James', 'Luka Dončić', 'Giannis Antetokounmpo', 'Stephen Curry', 'Nikola Jokić', 'DeMar DeRozan', 'Jimmy Butler', 'Darius Garland',
                'James Harden', 'Donovan Mitchell', 'Chris Paul', 'Fred VanVleet']


all_stars_20_21 = ['Giannis Antetokounmpo', 'Stephen Curry', 'Luka Dončić', 'LeBron James', 'Nikola Jokić', 'Jaylen Brown',
                'Paul George', 'Rudy Gobert', 'Damian Lillard', 'Chris Paul', 'Ben Simmons',
                'Bradley Beal', 'Kyrie Irving', 'Kawhi Leonard', 'Jayson Tatum', 'Zion Williamson', 'James Harden', 'Zach LaVine', 'Donovan Mitchell',
                'Julius Randle', 'Nikola Vučević', 'Kevin Durant', 'Anthony Davis', 'Joel Embiid']


all_stars_19_20 = ['Anthony Davis', 'Kawhi Leonard', 'Luka Dončić', 'Damian Lillard', 'James Harden', 'Ben Simmons',
                'Nikola Jokić', 'Jayson Tatum', 'Chris Paul', 'Russell Westbrook', 'Domantas Sabonis', 'LeBron James',
                'Giannis Antetokounmpo', 'Joel Embiid', 'Pascal Siakam', 'Kemba Walker', 'Trae Young', 'Khris Middleton', 'Bam Adebayo', 'Rudy Gobert',
                'Jimmy Butler', 'Kyle Lowry', 'Brandon Ingram', 'Donovan Mitchell']


## Retrieve data from Google Custom API Search
Through the Google Custom API Search Platform, I scraped 10 espn articles of each player per each season (19-20, 20-21, 21-22, 22-23, 23-24) and added them to their according data frame.

Limitation: the Free Trial Account only allows me to search 100 queries per min, so in my code, I implemented a cooldown period per min.

- Notice: After scrapping 19-20 with the same query clause, the table was highly incomplete (missing key players including: Stephen Curry / Lebron James / Kevin Durant). To resolve the issue, I chose to expand the query clause and scrape more information



> Retrieve 19-20 Season Player ESPN Headlines



In [ ]:
# Season period after:2019-10-22 before:2020-01-20

# clause for 19-20
clause = (
    '(takeaways OR analysis OR grades OR breakdown OR performance '
    'OR recap OR "game recap" OR reaction OR "instant reaction" '
    'OR observations OR "key observations" OR notebook '
    'OR "what we learned" OR "things we learned" '
    'OR "winners and losers" OR "winners, losers" '
    'OR report OR "postgame report" OR "game grades" OR "player grades")'
)

all_data_19_20 = []
chunk_size = 100  # given limit queries per min by Google Free Trial Account

def extract_date(snippet):
    match = re.search(r"(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec) \d{1,2}, \d{4}", snippet)
    return match.group(0) if match else None

for start in range(0, len(list_player_19_20), chunk_size):
    end = start + chunk_size
    players_chunk = list_player_19_20[start:end]
    print(f"Processing players {start} to {start + len(players_chunk) - 1} ...")

    for player in players_chunk:
        url = "https://www.googleapis.com/customsearch/v1"

        # fix error by limiting searches to NBA
        query = (
            f'"{player}" "NBA" '
            f'{clause} '
            f'site:espn.com/nba '
            f'after:2019-10-22 before:2020-01-20'
        )

        params = {
            "key": API_KEY_Isaac_2,
            "cx": CX,
            # define query below to maximize search for players performance
            "q": query,
            "num": 10,
            "lr": "lang_en"
        }
        response = requests.get(url, params=params).json()

        total_results_str = response.get("searchInformation", {}).get("totalResults", "0")
        try:
            total_results = int(total_results_str)
        except ValueError:
            total_results = 0

        for item in response.get("items", []):
            title = item.get("title")
            link = item.get("link")
            snippet = item.get("snippet", "")
            date = extract_date(snippet)

            all_data_19_20.append({
                "Player": player,
                "Title": title,
                "Link": link,
                "Date": date,
                "search_total_results": total_results
            })

    if end < len(list_player_19_20):
        print("Pausing 60 seconds for queries rate limit...")
        time.sleep(60)

df_19_20 = pd.DataFrame(all_data_19_20)
print(df_19_20)

# save NBA headlines to CSV
df_19_20.to_csv("nba_headlines_19_20.csv", index=False)
files.download("nba_headlines_19_20.csv")

Processing players 0 to 99 ...
Pausing 60 seconds for queries rate limit...
Processing players 100 to 199 ...
Pausing 60 seconds for queries rate limit...
Processing players 200 to 299 ...
Pausing 60 seconds for queries rate limit...
Processing players 300 to 399 ...
Pausing 60 seconds for queries rate limit...
Processing players 400 to 499 ...
Pausing 60 seconds for queries rate limit...
Processing players 500 to 529 ...
              Player                                              Title  \
0       James Harden  James Harden scores 60 points in 3 quarters of...   
1       James Harden  Rockets 158-111 Hawks (Nov 30, 2019) Game Reca...   
2       James Harden  James Harden is the greatest scorer of this NB...   
3       James Harden  Rockets hopeful NBA acts after James Harden du...   
4       James Harden  Sources: Rockets likely to file protest over m...   
...              ...                                                ...   
3884  League Average       Midseason grades for a

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



> Retrieve 20-21 Season Player ESPN Headlines




In [ ]:
# Season period after:2020-12-22 before:2021-02-16
all_data_20_21 = []
chunk_size = 100  # given limit queries per min by Google Free Trial Account

def extract_date(snippet):
    match = re.search(r"(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec) \d{1,2}, \d{4}", snippet)
    return match.group(0) if match else None

for start in range(0, len(list_player_20_21), chunk_size):
    end = start + chunk_size
    players_chunk = list_player_20_21[start:end]
    print(f"Processing players {start} to {start + len(players_chunk) - 1} ...")

    for player in players_chunk:
        url = "https://www.googleapis.com/customsearch/v1"

        # fix error by limiting searches to NBA
        query = (
            f'"{player}" "NBA" '
            f'(takeaways OR analysis OR grades OR breakdown) '
            f'site:espn.com/nba '
            f'after:2020-12-22 before:2021-02-16'
        )

        params = {
            "key": API_KEY_Isaac_2,
            "cx": CX,
            # define query below to maximize search for players performance
            "q": query,
            "num": 10,
            "lr": "lang_en"
        }
        response = requests.get(url, params=params).json()

        total_results_str = response.get("searchInformation", {}).get("totalResults", "0")
        try:
            total_results = int(total_results_str)
        except ValueError:
            total_results = 0

        for item in response.get("items", []):
            title = item.get("title")
            link = item.get("link")
            snippet = item.get("snippet", "")
            date = extract_date(snippet)

            all_data_20_21.append({
                "Player": player,
                "Title": title,
                "Link": link,
                "Date": date,
                "search_total_results": total_results
            })

    if end < len(list_player_20_21):
        print("Pausing 60 seconds for queries rate limit...")
        time.sleep(60)

df_20_21 = pd.DataFrame(all_data_20_21)
print(df_20_21)

# save NBA headlines to CSV
df_20_21.to_csv("nba_headlines_20_21.csv", index=False)
files.download("nba_headlines_20_21.csv")

Processing players 0 to 99 ...
Pausing 60 seconds for queries rate limit...
Processing players 100 to 199 ...
Pausing 60 seconds for queries rate limit...
Processing players 200 to 299 ...
Pausing 60 seconds for queries rate limit...
Processing players 300 to 399 ...
Pausing 60 seconds for queries rate limit...
Processing players 400 to 499 ...
Pausing 60 seconds for queries rate limit...
Processing players 500 to 540 ...
                Player                                              Title  \
0        Stephen Curry  How Stephen Curry's 'organized chaos' fuels hi...   
1        Stephen Curry  Golden State Warriors' Stephen Curry confident...   
2        Stephen Curry  Even when Stephen Curry is on, Warriors strugg...   
3        Stephen Curry  Golden State Warriors' Stephen Curry leaves te...   
4        Stephen Curry  Steve Kerr says Golden State Warriors won't st...   
..                 ...                                                ...   
729  Terrance Ferguson  Philadelphi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



> Retrieve 21-22 Season Player ESPN Headlines



In [ ]:
# Season period after:2021-10-19 before:2022-01-22
all_data_21_22 = []
chunk_size = 100  # given limit queries per min by Google Free Trial Account

def extract_date(snippet):
    match = re.search(r"(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec) \d{1,2}, \d{4}", snippet)
    return match.group(0) if match else None

for start in range(0, len(list_player_21_22), chunk_size):
    end = start + chunk_size
    players_chunk = list_player_21_22[start:end]
    print(f"Processing players {start} to {start + len(players_chunk) - 1} ...")

    for player in players_chunk:
        url = "https://www.googleapis.com/customsearch/v1"

        # fix error by limiting searches to NBA
        query = (
            f'"{player}" "NBA" '
            f'(takeaways OR analysis OR grades OR breakdown) '
            f'site:espn.com/nba '
            f'after:2021-10-19 before:2022-01-22'
        )

        params = {
            "key": API_KEY_Isaac_2,
            "cx": CX,
            # define query below to maximize search for players performance
            "q": query,
            "num": 10,
            "lr": "lang_en"
        }
        response = requests.get(url, params=params).json()

        total_results_str = response.get("searchInformation", {}).get("totalResults", "0")
        try:
            total_results = int(total_results_str)
        except ValueError:
            total_results = 0

        for item in response.get("items", []):
            title = item.get("title")
            link = item.get("link")
            snippet = item.get("snippet", "")
            date = extract_date(snippet)

            all_data_21_22.append({
                "Player": player,
                "Title": title,
                "Link": link,
                "Date": date,
                "search_total_results": total_results
            })

    if end < len(list_player_21_22):
        print("Pausing 60 seconds for queries rate limit...")
        time.sleep(60)

df_21_22 = pd.DataFrame(all_data_21_22)
print(df_21_22)

# save NBA headlines to CSV
df_21_22.to_csv("nba_headlines_21_22.csv", index=False)
files.download("nba_headlines_21_22.csv")

Processing players 0 to 99 ...
Pausing 60 seconds for queries rate limit...
Processing players 100 to 199 ...
Pausing 60 seconds for queries rate limit...
Processing players 200 to 299 ...
Pausing 60 seconds for queries rate limit...
Processing players 300 to 399 ...
Pausing 60 seconds for queries rate limit...
Processing players 400 to 499 ...
Pausing 60 seconds for queries rate limit...
Processing players 500 to 599 ...
Pausing 60 seconds for queries rate limit...
Processing players 600 to 605 ...
              Player                                              Title  \
0        Joel Embiid  Source: Philadelphia 76ers' Joel Embiid tests ...   
1        Joel Embiid  Josh Giddey wears Joel Embiid foul, puts up 19...   
2        Joel Embiid  'Unbelievable from the start,' Joel Embiid sco...   
3        Joel Embiid  Doc Rivers questions officials after no calls ...   
4        Joel Embiid  Chicago Bulls' Nikola Vucevic to miss multiple...   
...              ...                         

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



> Retrieve 22-23 Season Player ESPN Headlines



In [ ]:
# Season period after:2022-10-18 before:2023-01-21

all_data_22_23 = []
chunk_size = 100  # given limit queries per min by Google Free Trial Account

def extract_date(snippet):
    match = re.search(r"(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec) \d{1,2}, \d{4}", snippet)
    return match.group(0) if match else None

for start in range(0, len(list_player_22_23), chunk_size):
    end = start + chunk_size
    players_chunk = list_player_22_23[start:end]
    print(f"Processing players {start} to {start + len(players_chunk) - 1} ...")

    for player in players_chunk:
        url = "https://www.googleapis.com/customsearch/v1"

        # fix error by limiting searches to NBA
        query = (
            f'"{player}" "NBA" '
            f'(takeaways OR analysis OR grades OR breakdown) '
            f'site:espn.com/nba '
            f'after:2022-10-18 before:2023-01-21'
        )

        params = {
            "key": API_KEY_Isaac_2,
            "cx": CX,
            # define query below to maximize search for players performance
            "q": query,
            "num": 10,
            "lr": "lang_en"
        }
        response = requests.get(url, params=params).json()

        total_results_str = response.get("searchInformation", {}).get("totalResults", "0")
        try:
            total_results = int(total_results_str)
        except ValueError:
            total_results = 0

        for item in response.get("items", []):
            title = item.get("title")
            link = item.get("link")
            snippet = item.get("snippet", "")
            date = extract_date(snippet)

            all_data_22_23.append({
                "Player": player,
                "Title": title,
                "Link": link,
                "Date": date,
                "search_total_results": total_results
            })

    if end < len(list_player_22_23):
        print("Pausing 60 seconds for queries rate limit...")
        time.sleep(60)

df_22_23 = pd.DataFrame(all_data_22_23)
print(df_22_23)

# save NBA headlines to CSV
df_22_23.to_csv("nba_headlines_22_23.csv", index=False)
files.download("nba_headlines_22_23.csv")

Processing players 0 to 99 ...
Pausing 60 seconds for queries rate limit...
Processing players 100 to 199 ...
Pausing 60 seconds for queries rate limit...
Processing players 200 to 299 ...
Pausing 60 seconds for queries rate limit...
Processing players 300 to 399 ...
Pausing 60 seconds for queries rate limit...
Processing players 400 to 499 ...
Pausing 60 seconds for queries rate limit...
Processing players 500 to 539 ...
                Player                                              Title  \
0          Joel Embiid  Sixers' Joel Embiid out at least 2 games with ...   
1          Joel Embiid  NBA Opening Night live highlights, reactions a...   
2          Joel Embiid  Tyrese Maxey sets career high; 76ers win witho...   
3          Joel Embiid  Russell Westbrook claims Joel Embiid fouled hi...   
4          Joel Embiid  Big 76ers questions: With James Harden out, he...   
...                ...                                                ...   
1039        JD Davison  NBA 2022-23

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



> Retrieve 23-24 Season Player ESPN Healines



In [ ]:
# Season period after:2023-10-24 before:2024-01-20

all_data_23_24 = []
chunk_size = 100  # given limit queries per min by Google Free Trial Account

def extract_date(snippet):
    match = re.search(r"(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec) \d{1,2}, \d{4}", snippet)
    return match.group(0) if match else None

for start in range(0, len(list_player_23_24), chunk_size):
    end = start + chunk_size
    players_chunk = list_player_23_24[start:end]
    print(f"Processing players {start} to {start + len(players_chunk) - 1} ...")

    for player in players_chunk:
        url = "https://www.googleapis.com/customsearch/v1"

        # fix error by limiting searches to NBA
        query = (
            f'"{player}" "NBA" '
            f'(takeaways OR analysis OR grades OR breakdown) '
            f'site:espn.com/nba '
            f'after:2023-10-24 before:2024-01-20'
        )

        params = {
            "key": API_KEY_Isaac_2,
            "cx": CX,
            # define query below to maximize search for players performance
            "q": query,
            "num": 10,
            "lr": "lang_en"
        }
        response = requests.get(url, params=params).json()

        total_results_str = response.get("searchInformation", {}).get("totalResults", "0")
        try:
            total_results = int(total_results_str)
        except ValueError:
            total_results = 0

        for item in response.get("items", []):
            title = item.get("title")
            link = item.get("link")
            snippet = item.get("snippet", "")
            date = extract_date(snippet)

            all_data_23_24.append({
                "Player": player,
                "Title": title,
                "Link": link,
                "Date": date,
                "search_total_results": total_results
            })

    if end < len(list_player_23_24):
        print("Pausing 60 seconds for queries rate limit...")
        time.sleep(60)

df_23_24 = pd.DataFrame(all_data_23_24)
print(df_23_24)

# save NBA headlines to CSV
df_23_24.to_csv("nba_headlines_23_24.csv", index=False)
files.download("nba_headlines_23_24.csv")

Processing players 0 to 99 ...
Pausing 60 seconds for queries rate limit...
Processing players 100 to 199 ...
Pausing 60 seconds for queries rate limit...
Processing players 200 to 299 ...
Pausing 60 seconds for queries rate limit...
Processing players 300 to 399 ...
Pausing 60 seconds for queries rate limit...
Processing players 400 to 499 ...
Pausing 60 seconds for queries rate limit...
Processing players 500 to 572 ...
               Player                                              Title  \
0         Joel Embiid  Joel Embiid counters James Harden: Sixers let ...   
1         Joel Embiid  A new rule is changing the NBA's awards race; ...   
2         Joel Embiid  Joel Embiid scores 41, outduels Nikola Jokic a...   
3         Joel Embiid  Clips' brass says Harden comes to L.A. eager t...   
4         Joel Embiid  Paintings, silence and history: The 'final str...   
...               ...                                                ...   
1509  Frank Ntilikina  Hornets' LaMelo Bal

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Find Best ALPHA

In [ ]:
# remove print
def find_prediction(player_df, all_star_list, season_label, top_k=26):
    """
    player_df: DataFrame with columns ["Player", "final_sentiment_score", ...]
    all_star_list: list of true All-Star player names for that season
    """
    ranked = player_df.sort_values("final_sentiment_score", ascending=False)
    top_pred = set(ranked["Player"].head(top_k))
    true_allstars = set(all_star_list)

    tp = len(top_pred & true_allstars)
    fp = len(top_pred - true_allstars)
    fn = len(true_allstars - top_pred)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    union = top_pred | true_allstars
    iou = tp / len(union) if len(union) > 0 else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return precision, recall, f1, iou

In [ ]:
import numpy as np

def evaluate_alpha(alpha):
    """
    Recompute final_sentiment_score for each training season with a given alpha,
    then return average precision, recall, and F1 across seasons.
    """
    season_configs = [
        (player_media_features_19_20, all_stars_19_20, "2019–20"),
        (player_media_features_20_21, all_stars_20_21, "2020–21"),
        (player_media_features_21_22, all_stars_21_22, "2021–22"),
        (player_media_features_22_23, all_stars_22_23, "2022–23"),
    ]

    precisions = []
    recalls = []

    for df_season, all_star_list, season_label in season_configs:
        df_tmp = df_season.copy()

        df_tmp["final_sentiment_score"] = (
            alpha * df_tmp["popularity_component"]
            + (1.0 - alpha) * df_tmp["sentiment_component"]
        )

        # Optional: normalize so scores stay in [0,1] but keep ranking
        max_score = df_tmp["final_sentiment_score"].max()
        if max_score > 0:
            df_tmp["final_sentiment_score"] = df_tmp["final_sentiment_score"] / max_score

        k = len(all_star_list)
        precision, recall, f1, iou = find_prediction(
            df_tmp,
            all_star_list,
            season_label,
            top_k=k,
        )

        precisions.append(precision)
        recalls.append(recall)

    avg_precision = float(np.mean(precisions))
    avg_recall = float(np.mean(recalls))

    if avg_precision + avg_recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * avg_precision * avg_recall / (avg_precision + avg_recall)

    return avg_precision, avg_recall, f1


alphas = np.arange(0.01, 1.01, 0.01)

best_alpha = None
best_f1 = -1.0
best_precision = None
best_recall = None

for alpha in alphas:
    avg_precision, avg_recall, f1 = evaluate_alpha(alpha)

    if f1 > best_f1:
        best_f1 = f1
        best_alpha = alpha
        best_precision = avg_precision
        best_recall = avg_recall

print(f"Best alpha: {best_alpha:.2f}")
print(f"Best avg precision@k: {best_precision:.4f}")
print(f"Best avg recall@k:    {best_recall:.4f}")
print(f"Best avg F1:          {best_f1:.4f}")

NameError: name 'player_media_features_20_21' is not defined

Above it shows error. This would be resolved if I rerun the scarping for year 20_21. But I cannot do it, since I exceeded my daily limit of 10,000 queries per day. I am willing to demonstarte if

## Extract Sentiment Score for Each Player
Since NBA-all star is popularity voting, a player who have less than 10 articles rerturned from the start of season till the deadline of NBA-all star voting should be identified as unpopular. Therefore, recieving a less score. In the dataFrame we represent the sentiment score as final_sentiment_score.

> S (Sentiment component) = the avg sentiment score of the 10 articles -> [0,1]


> P (Popularity component) = the total search result returned by each query (player name)

> final_sentiment_score = ALPHA * popularity_component + (1 - ALPHA) * sentiment_component






In [ ]:
nltk.download("vader_lexicon")
sia = SentimentIntensityAnalyzer()
ALPHA = 0.87

cols_to_keep = [
    "Player",
    "media_sent_mean",
    "search_total_results",
    "sentiment_component",
    "popularity_component",
    "final_sentiment_score",
]

def sentiment_features(group):
    scores = group["sentiment"].values
    n = len(scores)

    if n == 0:
        return pd.Series({
            "media_sent_mean": 0.0,
            "media_sent_median": 0.0,
            "media_sent_pos_rate": 0.0,
            "media_sent_neg_rate": 0.0,
            "media_article_count": 0,
            "search_total_results": 0.0,
            "sentiment_component": 0.0,
            "popularity_component": 0.0,
            "final_sentiment_score": 0.0,
        })

    mean_score = scores.mean()
    median_score = np.median(scores)
    pos_rate = np.mean(scores > 0.2)
    neg_rate = np.mean(scores < -0.2)

    sentiment_component = (mean_score + 1.0) / 2.0
    sentiment_component = float(np.clip(sentiment_component, 0.0, 1.0))

    total_results = group["search_total_results"].max()
    if pd.isna(total_results):
        total_results = 0.0
    total_results = float(total_results)

    raw_log = np.log10(total_results + 1.0)
    popularity_component = min(raw_log / 4.0, 1.0)

    final_score =  ALPHA *  popularity_component + (1 - ALPHA) * sentiment_component

    return pd.Series({
        "media_sent_mean": mean_score,
        "media_sent_median": median_score,
        "media_sent_pos_rate": pos_rate,
        "media_sent_neg_rate": neg_rate,
        "media_article_count": int(n),
        "search_total_results": total_results,
        "sentiment_component": sentiment_component,
        "popularity_component": popularity_component,
        "final_sentiment_score": final_score,
    })

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!




> Extract Sentiment Score for Season 19-20



In [ ]:
df_19_20["text_for_sent"] = df_19_20["Title"]
df_19_20["sentiment"] = df_19_20["text_for_sent"].apply(
    lambda s: sia.polarity_scores(str(s))["compound"]
)

player_media_features_19_20 = df_19_20.groupby("Player").apply(sentiment_features).reset_index()
player_media_features_19_20 = player_media_features_19_20[cols_to_keep]
player_media_features_19_20.head(20)

max_score_19_20 = player_media_features_19_20["final_sentiment_score"].max()
# normalize
player_media_features_19_20["final_sentiment_score"] = player_media_features_19_20["final_sentiment_score"] / max_score_19_20

# transform player for each row, and their sentiment score
player_media_features_19_20.to_csv("espn_sentiment_score_19_20.csv", index=False)
files.download("espn_sentiment_score_19_20.csv")

/tmp/ipython-input-402686704.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  player_media_features_19_20 = df_19_20.groupby("Player").apply(sentiment_features).reset_index()


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



> Extract Sentiment Score for Season 20-21



In [ ]:
df_20_21["text_for_sent"] = df_20_21["Title"]
df_20_21["sentiment"] = df_20_21["text_for_sent"].apply(
    lambda s: sia.polarity_scores(str(s))["compound"]
)

player_media_features_20_21 = df_20_21.groupby("Player").apply(sentiment_features).reset_index()
player_media_features_20_21 = player_media_features_20_21[cols_to_keep]
player_media_features_20_21.head(20)

max_score_20_21 = player_media_features_20_21["final_sentiment_score"].max()
# normalize
player_media_features_20_21["final_sentiment_score"] = player_media_features_20_21["final_sentiment_score"] / max_score_20_21

# transform player for each row, and their sentiment score
player_media_features_20_21.to_csv("espn_sentiment_score_20_21.csv", index=False)
files.download("espn_sentiment_score_20_21.csv")

/tmp/ipython-input-857991182.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  player_media_features_20_21 = df_20_21.groupby("Player").apply(sentiment_features).reset_index()


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



> Extract Sentiment Score for Season 21-22



In [ ]:
df_21_22["text_for_sent"] = df_21_22["Title"]
df_21_22["sentiment"] = df_21_22["text_for_sent"].apply(
    lambda s: sia.polarity_scores(str(s))["compound"]
)

player_media_features_21_22 = df_21_22.groupby("Player").apply(sentiment_features).reset_index()
player_media_features_21_22 = player_media_features_21_22[cols_to_keep]
player_media_features_21_22.head(20)

max_score_21_22 = player_media_features_21_22["final_sentiment_score"].max()
# normalize
player_media_features_21_22["final_sentiment_score"] = player_media_features_21_22["final_sentiment_score"] / max_score_21_22

# transform player for each row, and their sentiment score
player_media_features_21_22.to_csv("espn_sentiment_score_21_22.csv", index=False)
files.download("espn_sentiment_score_21_22.csv")

/tmp/ipython-input-1472376686.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  player_media_features_21_22 = df_21_22.groupby("Player").apply(sentiment_features).reset_index()


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



> Extract Sentiment Score for Season 22-23



In [ ]:
df_22_23["text_for_sent"] = df_22_23["Title"]
df_22_23["sentiment"] = df_22_23["text_for_sent"].apply(
    lambda s: sia.polarity_scores(str(s))["compound"]
)

player_media_features_22_23 = df_22_23.groupby("Player").apply(sentiment_features).reset_index()
player_media_features_22_23 = player_media_features_22_23[cols_to_keep]
player_media_features_22_23.head(20)

max_score_22_23 = player_media_features_22_23["final_sentiment_score"].max()
# normalize
player_media_features_22_23["final_sentiment_score"] = player_media_features_22_23["final_sentiment_score"] / max_score_22_23

# transform player for each row, and their sentiment score
player_media_features_22_23.to_csv("espn_sentiment_score_22_23.csv", index=False)
files.download("espn_sentiment_score_22_23.csv")

/tmp/ipython-input-646200494.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  player_media_features_22_23 = df_22_23.groupby("Player").apply(sentiment_features).reset_index()


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



> Extract Sentiment Score for Season 23-24



In [ ]:
df_23_24["text_for_sent"] = df_23_24["Title"]
df_23_24["sentiment"] = df_23_24["text_for_sent"].apply(
    lambda s: sia.polarity_scores(str(s))["compound"]
)

player_media_features_23_24 = df_23_24.groupby("Player").apply(sentiment_features).reset_index()
player_media_features_23_24 = player_media_features_23_24[cols_to_keep]
player_media_features_23_24.head(20)

max_score_23_24 = player_media_features_23_24["final_sentiment_score"].max()
# normalize
player_media_features_23_24["final_sentiment_score"] = player_media_features_23_24["final_sentiment_score"] / max_score_23_24

# transform player for each row, and their sentiment score
player_media_features_23_24.to_csv("espn_sentiment_score_23_24.csv", index=False)
files.download("espn_sentiment_score_23_24.csv")

/tmp/ipython-input-3614473363.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  player_media_features_23_24 = df_23_24.groupby("Player").apply(sentiment_features).reset_index()


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Export Training / Test Final Results

In [ ]:
player_media_features_19_20["season"] = "2019-2020"
player_media_features_20_21["season"] = "2020-2021"
player_media_features_21_22["season"] = "2021-2022"
player_media_features_22_23["season"] = "2022-2023"
player_media_features_23_24["season"] = "2023-2024"

train_final_scores = pd.concat(
    [
        player_media_features_19_20,
        player_media_features_20_21,
        player_media_features_21_22,
        player_media_features_22_23,
    ],
    ignore_index=True
)

print(train_final_scores.shape)
print(train_final_scores.head())

train_final_scores.to_csv("espn_final_scores_train.csv", index=False)
files.download("espn_final_scores_train.csv")

test_final_scores = player_media_features_23_24.copy()

print(test_final_scores.shape)
print(test_final_scores.head())

test_final_scores.to_csv("espn_final_scores_test_23_24.csv", index=False)
files.download("espn_final_scores_test_23_24.csv")

(1348, 7)
              Player  media_sent_mean  search_total_results  \
0       Aaron Gordon         0.061240                  24.0   
1      Aaron Holiday         0.020230                  10.0   
2        Abdel Nader         0.000000                  10.0   
3        Adam Mokoka         0.000000                   3.0   
4  Admiral Schofield        -0.015617                   6.0   

   sentiment_component  popularity_component  final_sentiment_score     season  
0             0.530620              0.349485               0.515253  2019-2020  
1             0.510115              0.260348               0.398394  2019-2020  
2             0.500000              0.260348               0.397112  2019-2020  
3             0.500000              0.150515               0.256324  2019-2020  
4             0.492192              0.211275               0.333218  2019-2020  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

(370, 7)
          Player  media_sent_mean  search_total_results  sentiment_component  \
0     A.J. Green         0.000000                   3.0             0.500000   
1    A.J. Lawson         0.000000                   3.0             0.500000   
2   Aaron Gordon         0.246410                  23.0             0.623205   
3  Aaron Nesmith        -0.095450                   4.0             0.452275   
4     Al Horford         0.130575                   4.0             0.565287   

   popularity_component  final_sentiment_score     season  
0              0.150515               0.291355  2023-2024  
1              0.150515               0.291355  2023-2024  
2              0.345053               0.592556  2023-2024  
3              0.174743               0.319778  2023-2024  
4              0.174743               0.336063  2023-2024  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Verify on training set (19-20 , 20-21, 21-22, 22-23)
Metrics:
- Precision
- Recall
- IOU (Intersection over Union)

In [ ]:
def evaluate_allstar_predictions(player_df, all_star_list, season_label, top_k=26):
    """
    player_df: DataFrame with columns ["Player", "final_sentiment_score", ...]
    all_star_list: list of true All-Star player names for that season
    """
    ranked = player_df.sort_values("final_sentiment_score", ascending=False)
    top_pred = set(ranked["Player"].head(top_k))
    true_allstars = set(all_star_list)

    tp = len(top_pred & true_allstars)
    fp = len(top_pred - true_allstars)
    fn = len(true_allstars - top_pred)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    union = top_pred | true_allstars
    iou = tp / len(union) if len(union) > 0 else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    print(f"=== {season_label} ===")
    print(f"Top-{top_k} Precision: {precision:.3f}")
    print(f"Top-{top_k} Recall:    {recall:.3f}")
    print(f"Top-{top_k} F1:        {f1:.3f}")
    print(f"Top-{top_k} IoU:       {iou:.3f}")
    print(f"TP: {tp}, FP: {fp}, FN: {fn}")
    print()

    return precision, recall, f1, iou

In [ ]:
# training precision / recall
precision_19_20, recall_19_20, f1_19_20, iou_19_20 = evaluate_allstar_predictions(player_media_features_19_20, all_stars_19_20, "2019–20", top_k=25)
precision_20_21, recall_20_21, f1_20_21, iou_20_21 = evaluate_allstar_predictions(player_media_features_20_21, all_stars_20_21, "2020–21", top_k=27)
precision_21_22, recall_21_22, f1_21_22, iou_21_22 = evaluate_allstar_predictions(player_media_features_21_22, all_stars_21_22, "2021–22", top_k=27)
precision_22_23, recall_22_23, f1_22_23, iou_22_23 = evaluate_allstar_predictions(player_media_features_22_23, all_stars_22_23, "2022–23", top_k=27)
train_precision = (precision_19_20 + precision_20_21 + precision_21_22 + precision_22_23) / 4
train_recall    = (recall_19_20 + recall_20_21 + recall_21_22 + recall_22_23) / 4

print("Training Precision: " + str(round(train_precision, 2)))
print("Training Recall: "    + str(round(train_recall, 2)))

# testing precision / recall
precision_23_24, recall_23_24, f1_23_24, iou_23_24 = evaluate_allstar_predictions(player_media_features_23_24, all_stars_23_24, "2023–24", top_k=26)
print("Testing Precision: " + str(round(precision_23_24, 2)))
print("Testing Recall: "    + str(round(recall_23_24, 2)))

=== 2019–20 ===
Top-25 Precision: 0.480
Top-25 Recall:    0.480
Top-25 F1:        0.480
Top-25 IoU:       0.316
TP: 12, FP: 13, FN: 13

=== 2020–21 ===
Top-27 Precision: 0.704
Top-27 Recall:    0.704
Top-27 F1:        0.704
Top-27 IoU:       0.543
TP: 19, FP: 8, FN: 8

=== 2021–22 ===
Top-27 Precision: 0.593
Top-27 Recall:    0.593
Top-27 F1:        0.593
Top-27 IoU:       0.421
TP: 16, FP: 11, FN: 11

=== 2022–23 ===
Top-27 Precision: 0.556
Top-27 Recall:    0.556
Top-27 F1:        0.556
Top-27 IoU:       0.385
TP: 15, FP: 12, FN: 12

Training Precision: 0.58
Training Recall: 0.58
=== 2023–24 ===
Top-26 Precision: 0.615
Top-26 Recall:    0.615
Top-26 F1:        0.615
Top-26 IoU:       0.444
TP: 16, FP: 10, FN: 10

Testing Precision: 0.62
Testing Recall: 0.62


#Predicting on Current Season (25-26)

In [ ]:
list_player_25_26

# Season period after:2025-10-21 before:2025-12-01
all_data_25_26 = []
chunk_size = 100  # given limit queries per min by Google Free Trial Account

def extract_date(snippet):
    match = re.search(r"(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec) \d{1,2}, \d{4}", snippet)
    return match.group(0) if match else None

for start in range(0, len(list_player_25_26), chunk_size):
    end = start + chunk_size
    players_chunk = list_player_25_26[start:end]
    print(f"Processing players {start} to {start + len(players_chunk) - 1} ...")

    for player in players_chunk:
        url = "https://www.googleapis.com/customsearch/v1"

        # fix error by limiting searches to NBA
        query = (
            f'"{player}" "NBA" '
            f'(takeaways OR analysis OR grades OR breakdown) '
            f'site:espn.com/nba '
            f'after:2025-10-21 before:2025-12-01'
        )

        params = {
            "key": API_KEY_Isaac_2,
            "cx": CX,
            # define query below to maximize search for players performance
            "q": query,
            "num": 10,
            "lr": "lang_en"
        }
        response = requests.get(url, params=params).json()

        total_results_str = response.get("searchInformation", {}).get("totalResults", "0")
        try:
            total_results = int(total_results_str)
        except ValueError:
            total_results = 0

        for item in response.get("items", []):
            title = item.get("title")
            link = item.get("link")
            snippet = item.get("snippet", "")
            date = extract_date(snippet)

            all_data_25_26.append({
                "Player": player,
                "Title": title,
                "Link": link,
                "Date": date,
                "search_total_results": total_results
            })

    if end < len(list_player_25_26):
        print("Pausing 60 seconds for queries rate limit...")
        time.sleep(60)

df_25_26 = pd.DataFrame(all_data_25_26)
print(df_25_26)

# save NBA headlines to CSV
df_25_26.to_csv("nba_headlines_25_26.csv", index=False)
files.download("nba_headlines_25_26.csv")

# Extracting NBA 25-26 season:
df_25_26["text_for_sent"] = df_25_26["Title"]
df_25_26["sentiment"] = df_25_26["text_for_sent"].apply(
    lambda s: sia.polarity_scores(str(s))["compound"]
)

player_media_features_25_26 = df_25_26.groupby("Player").apply(sentiment_features).reset_index()
player_media_features_25_26 = player_media_features_25_26[cols_to_keep]
player_media_features_25_26.head(20)

max_score_25_26 = player_media_features_25_26["final_sentiment_score"].max()
player_media_features_25_26["final_sentiment_score"] = player_media_features_25_26["final_sentiment_score"] / max_score_25_26

player_media_features_25_26.to_csv("espn_sentiment_score_25_26.csv", index=False)
files.download("espn_sentiment_score_25_26.csv")

Processing players 0 to 99 ...
Pausing 60 seconds for queries rate limit...
Processing players 100 to 199 ...
Pausing 60 seconds for queries rate limit...
Processing players 200 to 299 ...
Pausing 60 seconds for queries rate limit...
Processing players 300 to 399 ...
Pausing 60 seconds for queries rate limit...
Processing players 400 to 490 ...
                Player                                              Title  \
0          Luka Dončić  Luka Doncic calls out slippery court after win...   
1          Luka Dončić  Lakers 121-111 Hornets (Nov 10, 2025) Game Rec...   
2          Luka Dončić  Luka Doncic scores 43 in Lakers' loss, says gr...   
3          Luka Dončić  NBA opening night: Biggest takeaways from Thun...   
4          Luka Dončić  Lakers rule out Luka Doncic, Austin Reaves vs....   
...                ...                                                ...   
2305  Christian Koloko  Hawks 122-102 Lakers (Nov 8, 2025) Game Recap ...   
2306       Chris Mañon  Inside a $241

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

/tmp/ipython-input-4131103413.py:76: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  player_media_features_25_26 = df_25_26.groupby("Player").apply(sentiment_features).reset_index()


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>